In [1]:

from pathlib import Path
import prism
import pandas as pd
import numpy as np
import xarray as xr
import scipy
import matplotlib.pyplot as plt
import warnings
import prism
import logging
import pickle

from imagematerials.model import GenericStocks, MaterialIntensities, GenericMaterials, SharesInflowStocks
from imagematerials.factory import ModelFactory, Sector
from imagematerials.preprocessing import get_preprocessing_data
from imagematerials.fossil_fuels.preprocessing.ffconstants import (path_base,STANDARD_SCEN_EXTERNAL_DATA)
from imagematerials.fossil_fuels.preprocessing.main import get_preprocessing_data_extraction
from imagematerials.fossil_fuels.preprocessing.main import get_preprocessing_data_processing
from imagematerials.fossil_fuels.preprocessing.main import get_preprocessing_data_transport
from imagematerials.fossil_fuels.preprocessing.main import get_preprocessing_data_pipelines
from imagematerials.fossil_fuels.preprocessing.main import get_preprocessing_data_storage
from imagematerials.fossil_fuels.preprocessing.ffconstants import REGIONS_TIMER

from imagematerials.util import reindex_material

In [2]:
YEAR_START = 1971   # start year of the simulation period
YEAR_END = 2100     # end year of the calculations
YEAR_OUT = 2100     # year of output generation = last year of reporting
year_end = 2100     # end year of the calculations
year_out = 2100
year_start = 1971 

#define path base or current path?
#Select scenario here and also in "ffconstants.py" as well 
circular_economy_config = None
climate_policy_config =   "SSP1_climate_policy"
scenario =  "SSP1_climate_policy"
base_dir =  Path(path_base,"data", "raw")


In [3]:

scenario_list = {
                # "SSP2":("SSP2_baseline", None), 
                 "SSP1_climate_policy":("SSP1_climate_policy", None), 
                #  "SSP1_delayed_action":("SSP1_delayed_action", None), 
                #  "SSP2_climate_policy":("SSP2_climate_policy", None), 
                #  "SSP2_delayed_action":("SSP2_delayed_action", None), 
                #  "SSP2_climate_policy_resource_efficiency":("SSP2_climate_policy_resource_efficiency", None),
                #  "SSP3_no_policy":("SSP3_no_policy", None)
                # add your scenarios
                 }

time_start = 1971
complete_timeline = prism.Timeline(time_start, YEAR_END, 1)
simulation_timeline = prism.Timeline(YEAR_START, YEAR_END, 1)

materials = [
    'aluminium', 'brick',  'cobalt', 'concrete','cement', 'copper', 'glass',        
    'lead', 'lithium', 'manganese', 'neodymium', 'nickel', 'pe', 'plastics',
    'rubber', 'sand', 'steel', 'tantalum', 'titanium', 'wood'
]



# Run model

In [ ]:

all_output = {}

# add debugging statements
logging.basicConfig(
    level=logging.DEBUG,
    format='%(name)s - %(levelname)s - %(message)s',
    force=True  # Override any existing config
)

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path(base_dir, "image", climate_scen)
    circular_economy_scenario_dir = None 
    scenario = base_dir /"fossil_fuels"/ STANDARD_SCEN_EXTERNAL_DATA

    # get preprocessing data

    # FOSSIL FUELS
    ff_sector = get_preprocessing_data("fossil_fuels", base_dir,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dir, cache = False) 
    ff_extr, ff_proc, ff_trp, ff_pipe, ff_str = (       
    {sec.name: sec for sec in ff_sector}[k]
    for k in ["ff_extr", "ff_proc", "ff_trp", "ff_pipe", "ff_str"])
    
    factory = ModelFactory(
   [ *ff_sector, *elc_sector], complete_timeline
    ).add(GenericStocks, ["ff_extr","ff_proc","ff_trp","ff_pipe", "ff_str",
                          "elc_grid_lines", "elc_grid_add", "elc_stor_phs"
                          ]
    ).add(SharesInflowStocks, ["elc_gen","elc_stor_other"]
    ).add(MaterialIntensities, ["ff_extr","ff_proc","ff_trp","ff_pipe", "ff_str", 
                         "elc_gen", "elc_grid_lines", "elc_grid_add", "elc_stor_phs", "elc_stor_other"
                          ])
    # ).add(GenericMaterials, ["ff_trp"])
    model = factory.finish()

    model.simulate(simulation_timeline)
    # save relevant outputs only to reduce memory usage
    all_output[scen_id] = model

    print(f"Finished {scen_id}")
 



Finished SSP1_climate_policy


In [ ]:

# all_output = {}

# # add debugging statements
# logging.basicConfig(
#     level=logging.DEBUG,
#     format='%(name)s - %(levelname)s - %(message)s',
#     force=True  # Override any existing config
# )

# for scen_id, (climate_scen, circular_scen) in scenario_list.items():
#     climate_policy_scenario_dir = Path(base_dir, "image", climate_scen)
#     circular_economy_scenario_dir = None 
#     scenario = base_dir /"fossil_fuels"/ STANDARD_SCEN_EXTERNAL_DATA

#     # get preprocessing data

#     # FOSSIL FUELS
#     ff_sector = get_preprocessing_data("fossil_fuels", base_dir,
#                                         climate_policy_scenario_dir, 
#                                         circular_economy_scenario_dir, cache = False) 
#     ff_extr, ff_proc, ff_trp, ff_pipe, ff_str = (       
#     {sec.name: sec for sec in ff_sector}[k]
#     for k in ["ff_extr", "ff_proc", "ff_trp", "ff_pipe", "ff_str"])
    

#     factory = ModelFactory(
#    [ *ff_sector], complete_timeline
#     ).add(GenericStocks, ["ff_extr","ff_proc","ff_trp","ff_pipe", "ff_str"]
#     ).add(MaterialIntensities, ["ff_extr","ff_proc","ff_trp","ff_pipe", "ff_str"])
#     # ).add(GenericMaterials, ["ff_trp"])
#     model = factory.finish()

#     model.simulate(simulation_timeline)
#     # save relevant outputs only to reduce memory usage
#     all_output[scen_id] = model
    
#     print(f"Finished {scen_id}")


In [ ]:
from pathlib import Path
import pickle

outdir = Path("/Users/sophiaschupp/Desktop/output")
outdir.mkdir(parents=True, exist_ok=True)

cache_file = outdir / f"{scen_id}.pkl"

to_save = {
    "inflow": model.inflow_materials,
    "stocks": model.stock_by_cohort_materials,
    "outflow": model.outflow_by_cohort_materials
}

with open(cache_file, "wb") as f:
    pickle.dump(to_save, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Saved to:", cache_file)   

# Import Knowledge graph

In [ ]:
# from imagematerials.concepts import create_fossil_fuel_graph

# kgraph_fossil_fuel = create_fossil_fuel_graph()
# stocks_ff_extr = model.ff_extr["stock_by_cohort"]

# test = kgraph_fossil_fuel.aggregate_sum(stocks_ff_extr, ["Coal", "Oil", "Gas"])

# from imagematerials.concepts import create_electricity_graph
# kgraph_electricity = create_electricity_graph()

list(model.elc_stor_other)

model.elc_stor_other["stock_by_cohort_materials"]

# stock_electricity_generation = model.elc_gen["stock_by_cohort"]



## Plot for materials embedded in each type of infrastructure (coal, oil, gas)

In [ ]:
logging.getLogger('matplotlib').setLevel(logging.ERROR)

stocks_ff_extr = model.ff_extr["stock_by_cohort_materials"]
stocks_ff_proc = model.ff_proc["stock_by_cohort_materials"]
stocks_ff_trp = model.ff_trp["stock_by_cohort_materials"]
stocks_ff_pipe = model.ff_pipe["stock_by_cohort_materials"]
stocks_ff_str = model.ff_str["stock_by_cohort_materials"]
all_sectors_ff_stocks = xr.concat([stocks_ff_extr, stocks_ff_proc, stocks_ff_trp, stocks_ff_pipe, stocks_ff_str], dim="Type")


test_all_FF_stocks = kgraph_fossil_fuel.aggregate_sum(all_sectors_ff_stocks, ["Coal", "Oil", "Gas"])
test_all_FF_stocks

# Stackedplot for coal stocks by material 
import matplotlib.pyplot as plt

colors = {
    "steel": "#d2edf3ff",
    "concrete": "#353992ff",
    "aluminium": "#f8c3daff",
    "copper": "#f05135ff",
    
}

unit = "megaton"

coal_materials = (
    test_all_FF_stocks
    .sel(Type="Coal")
    .sum(dim=["Region"])
    .sel(material=["aluminium", "copper", "steel", "concrete, "plastics"])
    .sel(Time=slice(1990, 2100))
    .pint.to(unit)
)

df_coal = coal_materials.to_pandas()

# df_coal.plot.area(stacked=True, figsize=(9, 6), color=[colors[col] for col in df_coal.columns])

# ax.title("Material Stock embedded in Coal Infrastructure - SSP3 No Policy")
# plt.xlabel("Time")
# plt.ylabel(f"Stock ({unit})")
# plt.grid(True, alpha=0.07)
# plt.legend(title="Material", loc="upper right")
# ax.spines['top'].set_visible(False)
# ax.spines['right'].set_visible(False)
# plt.show()
 
ax = df_coal.plot.area(
    stacked=True,
    figsize=(8, 6),
    color=[colors[col] for col in df_coal.columns]
)

ax.set_title("Material Stock embedded in Coal Infrastructure - SSP3 No Policy")
ax.set_xlabel("Time")
ax.set_ylabel(f"Stock ({unit})")

ax.grid(True, alpha=0.07)

ax.legend(title="Material", loc="upper left")

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.margins(x=0)

plt.show()

## Region division energy infrastructure

In [ ]:
logging.getLogger('matplotlib').setLevel(logging.ERROR)

stocks_ff_extr = model.ff_extr["stock_by_cohort_materials"]
stocks_ff_proc = model.ff_proc["stock_by_cohort_materials"]
stocks_ff_trp = model.ff_trp["stock_by_cohort_materials"]
stocks_ff_pipe = model.ff_pipe["stock_by_cohort_materials"]
stocks_ff_str = model.ff_str["stock_by_cohort_materials"]
stocks_elc_gen = model.elc_gen["stock_by_cohort_materials"]
stocks_elc_grid_lines = model.elc_grid_lines["stock_by_cohort_materials"]
stocks_elc_grid_add = model.elc_grid_add["stock_by_cohort_materials"]
stocks_elc_stor_phs = model.elc_stor_phs["stock_by_cohort_materials"]
stocks_elc_stor_other = model.elc_stor_other["stock_by_cohort_materials"]

all_sectors_all_stocks = xr.concat([stocks_ff_extr, stocks_ff_proc, stocks_ff_trp, stocks_ff_pipe, stocks_ff_str, stocks_elc_gen, stocks_elc_stor_phs, stocks_elc_grid_add, stocks_elc_grid_lines, stocks_elc_stor_other], dim="Type")

# inflow_ff_extr = model.ff_extr["inflow_materials"].to_array()
# inflow_ff_proc = model.ff_proc["inflow_materials"].to_array()
# inflow_ff_trp = model.ff_trp["inflow_materials"].to_array()
# inflow_ff_pipe = model.ff_pipe["inflow_materials"].to_array()
# inflow_ff_str = model.ff_str["inflow_materials"].to_array()
# inflow_elc_gen = model.elc_gen["inflow_materials"].to_array()
# inflow_elc_grid_lines = model.elc_grid_lines["inflow_materials"].to_array()
# inflow_elc_grid_add = model.elc_grid_add["inflow_materials"].to_array()
# inflow_elc_stor_phs = model.elc_stor_phs["inflow_materials"].to_array()
# inflow_elc_stor_other = model.elc_stor_other["inflow_materials"].to_array()

# all_sectors_all_inflow= xr.concat([inflow_ff_extr, inflow_ff_proc, inflow_ff_trp, inflow_ff_pipe, inflow_ff_str, inflow_elc_gen, inflow_elc_stor_phs, inflow_elc_grid_add, inflow_elc_grid_lines, inflow_elc_stor_other], dim="Type")


# print(all_sectors_all_stocks.dims)
# list(all_sectors_all_stocks.coords)
# print(all_sectors_all_stocks.coords)
# print(all_sectors_all_stocks.coords["Type"].values)

# all_sectors_ff_stocks = xr.concat([stocks_ff_extr, stocks_ff_proc, stocks_ff_trp, stocks_ff_pipe, stocks_ff_str].sum(dim="Type"))
# test_all_FF_stocks = kgraph_fossil_fuel.aggregate_sum(all_sectors_ff_stocks, ["Coal", "Oil", "Gas"])

# all_sectors_elc_stocks = xr.concat([stocks_elc_gen,stocks_elc_grid_add,stocks_elc_stor_phs], dim="Type")
# test_category_elc_stocks = kgraph_electricity.aggregate_sum(all_sectors_elc_stocks, ["Solar", "Wind", "Hydrogen", "Fossil", "Biomass", "Other Renewables", "Fossil + CCS", "Nuclear"])


import logging
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr

logging.getLogger('matplotlib').setLevel(logging.ERROR)

# =========================================================
# Aggregate original regions into R10 regions
# =========================================================

r10_regions = {
    "Africa": ["EAF", "NAF", "RSAF", "SAF", "WAF"],

    "China+": ["CHN", "KOR"],

    "Latin America": ["BRA", "MEX", "RCAM", "RSAM"],

    "Middle East": ["ME"],

    "India+": ["RSAS", "INDIA"],

    "Europe": ["CEU", "TUR", "WEU"],

    "North America": ["CAN", "USA"],

    "Reforming Economies": ["RUS", "UKR", "STAN"],

    "Rest of Asia": ["INDO", "SEAS"],

    "Pacific OECD": ["JAP", "OCE"]
}

# =========================================================
# Colors
# =========================================================

r10_colors = {
    "Africa": "#e57657ff",
    "China+": "#7b7fcaff",
    "Latin America": "#edafb8ff",
    "Middle East": "#6b4e71ff",
    "India+": "#618b4aff",
    "Europe": "#72a6d3ff",
    "North America": "#5f021fff",
    "Reforming Economies": "#b1dae1ff",
    "Rest of Asia": "#952a57ff",
    "Pacific OECD": "#5d9a92ff",
}


stack_order = [
    "Africa",
    "China+",
    "Latin America",
    "Middle East",
    "India+",
    "Europe",
    "North America",
    "Reforming Economies",
    "Rest of Asia",
    "Pacific OECD",
]

# =========================================================
# Material stocks
# =========================================================

unit = "megaton"

all_materials = (
    all_sectors_all_stocks
    .sum(dim=["Type", "material"])
    .sel(Time=slice(1971, 2020))
    .pint.to(unit)
)

# =========================================================
# Group regions into R10
# =========================================================

grouped = {
    region_name: (
        all_materials
        .sel(Region=regions)
        .sum(dim="Region")
    )
    for region_name, regions in r10_regions.items()
}

# Convert to dataframe
df_all = pd.DataFrame({
    name: da.to_pandas()
    for name, da in grouped.items()
})

# Keep desired order
df_all = df_all[stack_order]

# =========================================================
# Plot
# =========================================================

ax = df_all.plot.area(
    stacked=True,
    figsize=(8, 6),
    linewidth=0,
    color=[r10_colors[c] for c in df_all.columns]
)

ax.set_title(
    "Regional Contribution to Bulk Material Inflow - SSP2 BL",
    fontsize=18,
    pad=20
)

ax.set_xlabel(
    "Time",
    fontsize=16,
    labelpad=15,
    fontweight='medium'
)

ax.set_ylabel(
    f"Stock ({unit})",
    fontsize=16,
    labelpad=15,
    fontweight='medium'
)

ax.grid(True, alpha=0.07)

handles, labels = ax.get_legend_handles_labels()

ax.legend(
    handles[::-1],
    labels[::-1],
    title="Region",
    # bbox_to_anchor=(1.02, 1),
    loc="upper left",
    # frameon=False
)

ax.tick_params(axis='both', labelsize=13)
# ax.legend().remove()
ax.margins(x=0)

plt.tight_layout()
plt.show()

## Plot for material stock embedded in all energy infrastructure

In [ ]:
logging.getLogger('matplotlib').setLevel(logging.ERROR)

stocks_ff_extr = model.ff_extr["stock_by_cohort_materials"]
stocks_ff_proc = model.ff_proc["stock_by_cohort_materials"]
stocks_ff_trp = model.ff_trp["stock_by_cohort_materials"]
stocks_ff_pipe = model.ff_pipe["stock_by_cohort_materials"]
stocks_ff_str = model.ff_str["stock_by_cohort_materials"]
stocks_elc_gen = model.elc_gen["stock_by_cohort_materials"]
stocks_elc_grid_lines = model.elc_grid_lines["stock_by_cohort_materials"]
stocks_elc_grid_add = model.elc_grid_add["stock_by_cohort_materials"]
stocks_elc_stor_phs = model.elc_stor_phs["stock_by_cohort_materials"]
stocks_elc_stor_other = model.elc_stor_other["stock_by_cohort_materials"]

all_sectors_all_stocks = xr.concat([stocks_ff_extr, stocks_ff_proc, stocks_ff_trp, stocks_ff_pipe, stocks_ff_str, stocks_elc_gen, stocks_elc_stor_phs, stocks_elc_grid_add, stocks_elc_grid_lines, stocks_elc_stor_other], dim="Type")
all_sectors_all_stocks

# print(all_sectors_all_stocks.dims)
# list(all_sectors_all_stocks.coords)
# print(all_sectors_all_stocks.coords)
# print(all_sectors_all_stocks.coords["Type"].values)

# all_sectors_ff_stocks = xr.concat([stocks_ff_extr, stocks_ff_proc, stocks_ff_trp, stocks_ff_pipe, stocks_ff_str].sum(dim="Type"))
# test_all_FF_stocks = kgraph_fossil_fuel.aggregate_sum(all_sectors_ff_stocks, ["Coal", "Oil", "Gas"])

# all_sectors_elc_stocks = xr.concat([stocks_elc_gen,stocks_elc_grid_add,stocks_elc_stor_phs], dim="Type")
# test_category_elc_stocks = kgraph_electricity.aggregate_sum(all_sectors_elc_stocks, ["Solar", "Wind", "Hydrogen", "Fossil", "Biomass", "Other Renewables", "Fossil + CCS", "Nuclear"])


# Stackedplot for coal stocks by material 
import matplotlib.pyplot as plt

colors = {
    "steel": "#d2edf3ff",
    "concrete": "#353992ff",
    "aluminium": "#f8c3daff",
    "copper": "#f05135ff"
}

unit = "megaton"

all_materials = (
    all_sectors_all_stocks
    .sum(dim=["Region", "Type"])
    .sel(material=["aluminium", "copper", "steel", "concrete"])
    .sel(Time=slice(1990, 2100))
    .pint.to(unit)
)

df_all = all_materials.to_pandas()

ax = df_all.plot.area(stacked=True, figsize=(8, 6), color=[colors[col] for col in df_all.columns],linewidth=0)
# df_all.plot.area(stacked=True, figsize=(8, 6))

ax.set_title("Material Stock embedded in ALL Energy Infrastructure - SSP2 BL")
ax.set_xlabel("Time", fontsize=19,labelpad=20, fontweight='medium')
ax.set_ylabel(f"Stock ({unit})", fontsize=19, labelpad=20,fontweight='medium')
ax.grid(True, alpha=0.07)

ax.tick_params(axis='both', labelsize=14)

# ax.spines['top'].set_visible(False)
# ax.spines['right'].set_visible(False)
# ax.legend(title="Material", loc="upper left")
ax.legend().remove()
ax.set_ylim(0, 62_000)
ax.margins(x=0)


plt.show()

## Plot with total stock per material per scenario (with output CSV)

In [ ]:
logging.getLogger('matplotlib').setLevel(logging.ERROR)

material = "aluminium"
unit = "megaton"
firstplotyear = 1990
lastplotyear = 2100

#plot for steel in all regions and types for each sector plotted on the same graph to compare the sectors.
stocks_ff_extr = model.ff_extr["stock_by_cohort_materials"].sel(material=material,Time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
stocks_ff_proc = model.ff_proc["stock_by_cohort_materials"].sel(material=material,Time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
stocks_ff_trp = model.ff_trp["stock_by_cohort_materials"].sel(material=material,Time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
stocks_ff_pipe = model.ff_pipe["stock_by_cohort_materials"].sel(material=material,Time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
stocks_ff_str = model.ff_str["stock_by_cohort_materials"].sel(material=material,Time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
stocks_elc_gen = model.elc_gen["stock_by_cohort_materials"].sel(material=material,Time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
stocks_elc_grid_lines = model.elc_grid_lines["stock_by_cohort_materials"].sel(material=material,Time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
stocks_elc_grid_add = model.elc_grid_add["stock_by_cohort_materials"].sel(material=material,Time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
stocks_elc_stor_phs = model.elc_stor_phs["stock_by_cohort_materials"].sel(material=material,Time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
stocks_elc_stor_other = model.elc_stor_other["stock_by_cohort_materials"].sel(material=material,Time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)

total_overall = stocks_ff_extr + stocks_ff_proc + + stocks_ff_trp + stocks_ff_pipe + stocks_ff_str + stocks_elc_gen + stocks_elc_stor_phs + stocks_elc_grid_add + stocks_elc_grid_lines + stocks_elc_stor_other
# total_overall = stocks_ff_extr + stocks_ff_proc + stocks_ff_str + stocks_elc_gen + stocks_elc_stor_phs + stocks_elc_grid_add + stocks_elc_grid_lines + stocks_elc_stor_other


fig, ax = plt.subplots(figsize=(8,6))

total_overall.plot(label="Total", linewidth=2, linestyle="--")
plt.title(f"{material.title()} Stock embedded in Energy Infrastructure - SSP2 DA", fontweight="bold")
plt.xlabel("Time")
plt.ylabel(f"Stock ({unit})", fontsize=14)
plt.grid(True)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.margins(x=0)

ax.legend(loc="upper left")
ax.grid(alpha=0.3)

from pathlib import Path

output_dir = Path("/Users/SophiaSchupp/Desktop/PreliminaryResultsFigures")

filename = f"ssp3_np_{material}_stock.csv"

total_overall.to_pandas().to_csv(output_dir / filename)

## plot for inflow and outflow of each material per scenario

In [ ]:
logging.getLogger('matplotlib').setLevel(logging.ERROR)

material = "aluminium"
unit = "megaton"
firstplotyear = 1990
lastplotyear = 2100


#plot for steel in all regions and types for each sector plotted on the same graph to compare the sectors.
inflow_ff_extr = model.ff_extr["inflow_materials"].to_array().sel(material=material, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
inflow_ff_proc = model.ff_proc["inflow_materials"].to_array().sel(material=material, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
inflow_ff_trp = model.ff_trp["inflow_materials"].to_array().sel(material=material, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
inflow_ff_pipe = model.ff_pipe["inflow_materials"].to_array().sel(material=material, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
inflow_ff_str = model.ff_str["inflow_materials"].to_array().sel(material=material, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
inflow_elc_gen = model.elc_gen["inflow_materials"].to_array().sel(material=material, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
inflow_elc_grid_lines = model.elc_grid_lines["inflow_materials"].to_array().sel(material=material, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
inflow_elc_grid_add = model.elc_grid_add["inflow_materials"].to_array().sel(material=material, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
inflow_elc_stor_phs = model.elc_stor_phs["inflow_materials"].to_array().sel(material=material, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
inflow_elc_stor_other = model.elc_stor_other["inflow_materials"].to_array().sel(material=material, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)


total_inflow_overall = inflow_ff_extr + inflow_ff_proc +inflow_ff_trp + inflow_ff_pipe + inflow_ff_str + inflow_elc_gen + inflow_elc_stor_phs + inflow_elc_grid_add + inflow_elc_grid_lines
# total_inflow_overall = inflow_ff_extr + inflow_ff_proc + inflow_ff_str + inflow_elc_gen + inflow_elc_stor_phs + inflow_elc_grid_add + inflow_elc_grid_lines + inflow_elc_stor_other

outflow_ff_extr = model.ff_extr["outflow_by_cohort_materials"].to_array().sel(material=material, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
outflow_ff_proc = model.ff_proc["outflow_by_cohort_materials"].to_array().sel(material=material, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
outflow_ff_trp = model.ff_trp["outflow_by_cohort_materials"].to_array().sel(material=material, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
outflow_ff_pipe = model.ff_pipe["outflow_by_cohort_materials"].to_array().sel(material=material, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
outflow_ff_str = model.ff_str["outflow_by_cohort_materials"].to_array().sel(material=material, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
outflow_elc_gen = model.elc_gen["outflow_by_cohort_materials"].to_array().sel(material=material, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
outflow_elc_grid_lines = model.elc_grid_lines["outflow_by_cohort_materials"].to_array().sel(material=material, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
outflow_elc_grid_add = model.elc_grid_add["outflow_by_cohort_materials"].to_array().sel(material=material, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
outflow_elc_stor_phs = model.elc_stor_phs["outflow_by_cohort_materials"].to_array().sel(material=material, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
outflow_elc_stor_other = model.elc_stor_other["outflow_by_cohort_materials"].to_array().sel(material=material, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)

total_outflow_overall = outflow_ff_extr + outflow_ff_proc + outflow_ff_trp + outflow_ff_pipe + outflow_ff_str + outflow_elc_gen + outflow_elc_stor_phs + outflow_elc_grid_add + outflow_elc_grid_lines
# total_outflow_overall = outflow_ff_extr +  outflow_ff_proc + outflow_ff_str + outflow_elc_gen + outflow_elc_stor_phs + outflow_elc_grid_add + outflow_elc_grid_lines + outflow_elc_stor_other

fig, ax = plt.subplots(figsize=(8,6))

total_inflow_overall.plot(label="Inflow", linewidth=2, linestyle="--")
(-total_outflow_overall).plot(
    label="Outflow",
    linewidth=2,
    linestyle="--"
)

plt.title(f"{material.title()} Flows embedded in Energy Infrastructure - SSP2 CP", fontweight="bold")
plt.xlabel("Time")
plt.ylabel(f"Stock ({unit})", fontsize=14)
plt.grid(True)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.margins(x=0)

ax.legend(loc="upper left")
ax.grid(alpha=0.3)

from pathlib import Path

output_dir = Path("/Users/SophiaSchupp/Desktop/PreliminaryResultsFigures")

from pathlib import Path

output_dir = Path("/Users/SophiaSchupp/Desktop/PreliminaryResultsFigures")

# save inflow
filename = f"ssp3_np_{material}_inflow.csv"
total_inflow_overall.to_pandas().to_csv(output_dir / filename)

# save outflow
filename = f"ssp3_np_{material}_outflow.csv"
total_outflow_overall.to_pandas().to_csv(output_dir / filename)



## Bar chart for materials embedded in different types of energy infrastructure

In [ ]:
logging.getLogger('matplotlib').setLevel(logging.ERROR)

stocks_ff_extr = model.ff_extr["stock_by_cohort_materials"]
stocks_ff_proc = model.ff_proc["stock_by_cohort_materials"]
stocks_ff_trp = model.ff_trp["stock_by_cohort_materials"]
stocks_ff_pipe = model.ff_pipe["stock_by_cohort_materials"]
stocks_ff_str = model.ff_str["stock_by_cohort_materials"]
stocks_elc_gen = model.elc_gen["stock_by_cohort_materials"]
stocks_elc_grid_lines = model.elc_grid_lines["stock_by_cohort_materials"]
stocks_elc_grid_add = model.elc_grid_add["stock_by_cohort_materials"]
stocks_elc_stor_other = model.elc_stor_other["stock_by_cohort_materials"]
stocks_elc_stor_phs = model.elc_stor_phs["stock_by_cohort_materials"]


all_sectors_all_stocks = xr.concat([stocks_ff_extr, stocks_ff_proc, stocks_ff_trp, stocks_ff_pipe, stocks_ff_str, stocks_elc_gen, stocks_elc_stor_phs, stocks_elc_grid_add, stocks_elc_grid_lines, stocks_elc_stor_other], dim="Type")
all_sectors_all_stocks


categories = {

    "Solar": ["SPV", "SPVR","CSP" ],

    "Wind": ["WON","WOFF"],

    "Other Renewables": ["WAVE", "OREN", "GEO","GeoCHP" ],

    "Hydropower": ["HYD"],

    "Hydrogen": [ "H2P", "H2CHP" ],

    "Nuclear": [ "NUC" ],

    "Fossil Fuels ": [

        "Coal Opencast",
        "Coal Underground",
        "Gas Offshore",
        "Gas Onshore",
        "Oil Offshore",
        "Oil Onshore",
        "Coal Preparation",
        "Gas Processing",
        "Oil Refinery",
        "Coal Inland Ship",
        "Coal Ocean Ship",
        "Coal Rail",
        "Coal Truck",
        "Gas Inland Ship",
        "Gas Ocean Ship",
        "Gas Rail",
        "Gas Truck",
        "Oil Inland Ship",
        "Oil Ocean Ship",
        "Oil Rail",
        "Oil Truck",
        "Gas Distribution Pipeline",
        "Gas Transmission Pipeline",
        "Oil Offshore Crude Pipeline",
        "Oil Onshore Crude Pipeline",
        "Oil Offshore Product Pipeline",
        "Oil Onshore Product Pipeline",
        "Oil Storage"],

    "Fossil Electricity": [
        "ClST",
        "OlST",
        "NGOT",
        "IGCC",
        "OlCC",
        "NGCC",
        "ClCHP",
        "OlCHP",
        "NGCHP"
    ],


    "Fossil + CCS": [
        "ClCS",
        "OlCS",
        "NGCS",
        "ClCHPCS",
        "OlCHPCS",
        "NGCHPCS"
    ],


    "Biomass": [
        "BioST",
        "BioCC",
        "BioCS",
        "BioCHP",
        "BioCHPCS"
    ],


   "Electricity Storage": [
        "PHS",
        "Compressed Air",
        "Deep-cycle Lead-Acid",
        "Flywheel",
        "Hydrogen FC",
        "LFP",
        "LMO",
        "LTO",
        "Lithium Ceramic",
        "Lithium Sulfur",
        "Lithium-air",
        "NCA",
        "NMC",
        "NiMH",
        "Sodium-Sulfur",
        "Vanadium Redox",
        "ZEBRA"
    ],

    "Electricity Grid": [
        "HV - Lines - Overhead",
        "HV - Lines - Underground",
        "MV - Lines - Overhead",
        "MV - Lines - Underground",
        "LV - Lines - Overhead",
        "LV - Lines - Underground",
        "HV - Transformers",
        "HV - Substations",
        "MV - Transformers",
        "MV - Substations",
        "LV - Transformers",
        "LV - Substations"
    ],

    # "Other": [
    #     "FREE12"
    # ]
}
category_colors = {

    "Solar": "#81a3aaff",             
    "Wind": "#9bb17cff",                
    "Other Renewables": "#619ed0ff",    
    "Hydropower": "#a5d8f7ff",
    "Hydrogen": "#416687ff",          
    "Nuclear": "#991d51ff",             
    "Fossil Fuels ": "#4a3f69ff",   
    "Fossil Electricity": "#885ba6ff",   
    "Fossil + CCS": "#b6a1dbff",          
    "Biomass": "#2f7840ff" ,              
    "Electricity Storage": "#da607eff",   
    "Electricity Grid": "#eccbe7ff",      
    # "Other": "#f8c6b2"                  
}


stack_order = [
    "Fossil Fuels ",
    "Fossil Electricity",
    "Fossil + CCS",
    "Electricity Storage",
    "Electricity Grid",
    "Nuclear",
    "Other Renewables",
    "Hydropower",
    "Biomass",
    "Wind",
    "Hydrogen",
    "Solar",
    # "Other"  
]

import matplotlib.pyplot as plt

unit = "megaton"

grouped = {
    category: (
        all_sectors_all_stocks
        .sel(Type=techs)
        .sel(material=["aluminium", "copper", "steel", "concrete"])
        .sum(dim=["Region", "material","Type"])
        .sel(Time=slice(1990, 2100))
        .pint.to(unit)
    )
    for category, techs in categories.items()
}

df_all = pd.DataFrame({
    name: da.to_pandas()
    for name, da in grouped.items()
})

df_2years = df_all.loc[[2050, 2100]].sort_index()
df_2years = df_2years[stack_order]

ax = df_2years.plot(
    kind="bar",
    stacked=True,
    figsize=(6,10),
    color=[category_colors[c] for c in df_2years.columns]
)

ax.set_title(
    "Contribution of each type of ALL Energy Infrastructure to Material Stocks - SSP1 CP Other"
)

ax.set_xlabel("Time", fontsize=19,labelpad=20, fontweight='medium')
ax.set_ylabel(f"Stock ({unit})", fontsize=19,labelpad=20, fontweight='medium')

ax.grid(True, alpha=0.07)
handles, labels = ax.get_legend_handles_labels()

ax.legend(
    handles[::-1],
    labels[::-1],
    title="Category"
)

# ax.spines['top'].set_visible(False)
# ax.spines['right'].set_visible(False)
ax.margins(x=0)
# ax.legend().remove()
ax.set_ylim(0, 65_000)
ax.legend().remove()

ax.tick_params(axis='both', labelsize=14)

plt.show()

## Stacked line chart for diff types of energy infrastructure

In [ ]:
logging.getLogger('matplotlib').setLevel(logging.ERROR)

stocks_ff_extr = model.ff_extr["stock_by_cohort_materials"]
stocks_ff_proc = model.ff_proc["stock_by_cohort_materials"]
stocks_ff_trp = model.ff_trp["stock_by_cohort_materials"]
stocks_ff_pipe = model.ff_pipe["stock_by_cohort_materials"]
stocks_ff_str = model.ff_str["stock_by_cohort_materials"]
stocks_elc_gen = model.elc_gen["stock_by_cohort_materials"]
stocks_elc_grid_lines = model.elc_grid_lines["stock_by_cohort_materials"]
stocks_elc_grid_add = model.elc_grid_add["stock_by_cohort_materials"]
stocks_elc_stor_phs = model.elc_stor_phs["stock_by_cohort_materials"]
stocks_elc_stor_other = model.elc_stor_other["stock_by_cohort_materials"]

# print(stocks_elc_stor_other.Type)

all_sectors_all_stocks = xr.concat([stocks_ff_extr, stocks_ff_proc, stocks_ff_trp, stocks_ff_pipe, stocks_ff_str, stocks_elc_gen, stocks_elc_stor_phs, stocks_elc_grid_add, stocks_elc_grid_lines, stocks_elc_stor_other], dim="Type")


categories = {

    "Solar": ["SPV", "SPVR","CSP" ],

    "Wind": ["WON","WOFF"],

    "Other Renewables": ["WAVE", "OREN", "GEO","GeoCHP" ],

    "Hydropower": ["HYD"],

    "Hydrogen": [ "H2P", "H2CHP" ],

    "Nuclear": [ "NUC" ],

    "Fossil Fuels ": [

        "Coal Opencast",
        "Coal Underground",
        "Gas Offshore",
        "Gas Onshore",
        "Oil Offshore",
        "Oil Onshore",
        "Coal Preparation",
        "Gas Processing",
        "Oil Refinery",
        "Coal Inland Ship",
        "Coal Ocean Ship",
        "Coal Rail",
        "Coal Truck",
        "Gas Inland Ship",
        "Gas Ocean Ship",
        "Gas Rail",
        "Gas Truck",
        "Oil Inland Ship",
        "Oil Ocean Ship",
        "Oil Rail",
        "Oil Truck",
        "Gas Distribution Pipeline",
        "Gas Transmission Pipeline",
        "Oil Offshore Crude Pipeline",
        "Oil Onshore Crude Pipeline",
        "Oil Offshore Product Pipeline",
        "Oil Onshore Product Pipeline",
        "Oil Storage"],

    "Fossil Electricity": [
        "ClST",
        "OlST",
        "NGOT",
        "IGCC",
        "OlCC",
        "NGCC",
        "ClCHP",
        "OlCHP",
        "NGCHP"
    ],


    "Fossil + CCS": [
        "ClCS",
        "OlCS",
        "NGCS",
        "ClCHPCS",
        "OlCHPCS",
        "NGCHPCS"
    ],


    "Biomass": [
        "BioST",
        "BioCC",
        "BioCS",
        "BioCHP",
        "BioCHPCS"
    ],


    "Electricity Storage": [
        "PHS",
        "Compressed Air",
        "Deep-cycle Lead-Acid",
        "Flywheel",
        "Hydrogen FC",
        "LFP",
        "LMO",
        "LTO",
        "Lithium Ceramic",
        "Lithium Sulfur",
        "Lithium-air",
        "NCA",
        "NMC",
        "NiMH",
        "Sodium-Sulfur",
        "Vanadium Redox",
        "ZEBRA"
    ],

    "Electricity Grid": [
        "HV - Lines - Overhead",
        "HV - Lines - Underground",
        "MV - Lines - Overhead",
        "MV - Lines - Underground",
        "LV - Lines - Overhead",
        "LV - Lines - Underground",
        "HV - Transformers",
        "HV - Substations",
        "MV - Transformers",
        "MV - Substations",
        "LV - Transformers",
        "LV - Substations"
    ],

    # "Other": [
    #     "FREE12"
    # ]
}
category_colors = {

    "Solar": "#81a3aaff",             
    "Wind": "#9bb17cff",                
    "Other Renewables": "#619ed0ff",    
    "Hydropower": "#a5d8f7ff",
    "Hydrogen": "#416687ff",          
    "Nuclear": "#991d51ff",             
    "Fossil Fuels ": "#4a3f69ff",   
    "Fossil Electricity": "#885ba6ff",   
    "Fossil + CCS": "#b6a1dbff",          
    "Biomass": "#2f7840ff" ,              
    "Electricity Storage": "#da607eff",   
    "Electricity Grid": "#eccbe7ff",      
    # "Other": "#f8c6b2"                  
}


stack_order = [
    "Fossil Fuels ",
    "Fossil Electricity",
    "Fossil + CCS",
    "Electricity Storage",
    "Electricity Grid",
    "Nuclear",
    "Other Renewables",
    "Hydropower",
    "Biomass",
    "Wind",
    "Hydrogen",
    "Solar",
    # "Other"  
]

import matplotlib.pyplot as plt

unit = "megaton"

grouped = {
    category: (
        all_sectors_all_stocks
        .sel(Type=techs)
        # .sel(material=["aluminium", "copper", "steel", "concrete"])
        # .sel(material=["concrete"])
        .sel(material=["steel"])
        # .sel(material=["aluminium", "copper"])
        .sum(dim=["Region", "material","Type"])
        .sel(Time=slice(1971, 2020))
        .pint.to(unit)
    )
    for category, techs in categories.items()
}

df_all = pd.DataFrame({
    name: da.to_pandas()
    for name, da in grouped.items()
})

df_all = df_all[stack_order]
ax = df_all.plot.area(stacked=True, figsize=(8, 6), color=[category_colors[c] for c in df_all.columns])

ax.set_title(
    "Contribution of each type of ALL Energy Infrastructure to Steel Stocks - Historical (SSP1-CP)"
)

ax.set_xlabel("Time", fontsize=19,labelpad=20, fontweight='medium')
ax.set_ylabel(f"Stock ({unit})", fontsize=19,labelpad=20, fontweight='medium')

ax.grid(True, alpha=0.07)
handles, labels = ax.get_legend_handles_labels()

ax.legend(
    handles[::-1],
    labels[::-1],
    title="Category"
)

# ax.spines['top'].set_visible(False)
# ax.spines['right'].set_visible(False)
ax.margins(x=0)
# ax.legend().remove()
# ax.set_ylim(0, 63_000)

ax.tick_params(axis='both', labelsize=14)

plt.show()

In [ ]:
logging.getLogger('matplotlib').setLevel(logging.ERROR)

inflow_ff_extr = model.ff_extr["inflow_materials"].to_array()
inflow_ff_proc = model.ff_proc["inflow_materials"].to_array()
inflow_ff_trp = model.ff_trp["inflow_materials"].to_array()
inflow_ff_pipe = model.ff_pipe["inflow_materials"].to_array()
inflow_ff_str = model.ff_str["inflow_materials"].to_array()
inflow_elc_gen = model.elc_gen["inflow_materials"].to_array()
inflow_elc_grid_lines = model.elc_grid_lines["inflow_materials"].to_array()
inflow_elc_grid_add = model.elc_grid_add["inflow_materials"].to_array()
inflow_elc_stor_phs = model.elc_stor_phs["inflow_materials"].to_array()
inflow_elc_stor_other = model.elc_stor_other["inflow_materials"].to_array()

all_sectors_all_inflow= xr.concat([inflow_ff_extr, inflow_ff_proc, inflow_ff_trp, inflow_ff_pipe, inflow_ff_str, inflow_elc_gen, inflow_elc_stor_phs, inflow_elc_grid_add, inflow_elc_grid_lines, inflow_elc_stor_other], dim="Type")


categories = {

    "Solar": ["SPV", "SPVR","CSP" ],

    "Wind": ["WON","WOFF"],

    "Other Renewables": ["WAVE", "OREN", "GEO","GeoCHP" ],

    "Hydropower": ["HYD"],

    "Hydrogen": [ "H2P", "H2CHP" ],

    "Nuclear": [ "NUC" ],

    "Fossil Fuels ": [

        "Coal Opencast",
        "Coal Underground",
        "Gas Offshore",
        "Gas Onshore",
        "Oil Offshore",
        "Oil Onshore",
        "Coal Preparation",
        "Gas Processing",
        "Oil Refinery",
        "Coal Inland Ship",
        "Coal Ocean Ship",
        "Coal Rail",
        "Coal Truck",
        "Gas Inland Ship",
        "Gas Ocean Ship",
        "Gas Rail",
        "Gas Truck",
        "Oil Inland Ship",
        "Oil Ocean Ship",
        "Oil Rail",
        "Oil Truck",
        "Gas Distribution Pipeline",
        "Gas Transmission Pipeline",
        "Oil Offshore Crude Pipeline",
        "Oil Onshore Crude Pipeline",
        "Oil Offshore Product Pipeline",
        "Oil Onshore Product Pipeline",
        "Oil Storage"],

    "Fossil Electricity": [
        "ClST",
        "OlST",
        "NGOT",
        "IGCC",
        "OlCC",
        "NGCC",
        "ClCHP",
        "OlCHP",
        "NGCHP"
    ],


    "Fossil + CCS": [
        "ClCS",
        "OlCS",
        "NGCS",
        "ClCHPCS",
        "OlCHPCS",
        "NGCHPCS"
    ],


    "Biomass": [
        "BioST",
        "BioCC",
        "BioCS",
        "BioCHP",
        "BioCHPCS"
    ],


   "Electricity Storage": [
        "PHS",
        "Compressed Air",
        "Deep-cycle Lead-Acid",
        "Flywheel",
        "Hydrogen FC",
        "LFP",
        "LMO",
        "LTO",
        "Lithium Ceramic",
        "Lithium Sulfur",
        "Lithium-air",
        "NCA",
        "NMC",
        "NiMH",
        "Sodium-Sulfur",
        "Vanadium Redox",
        "ZEBRA"
    ],

    "Electricity Grid": [
        "HV - Lines - Overhead",
        "HV - Lines - Underground",
        "MV - Lines - Overhead",
        "MV - Lines - Underground",
        "LV - Lines - Overhead",
        "LV - Lines - Underground",
        "HV - Transformers",
        "HV - Substations",
        "MV - Transformers",
        "MV - Substations",
        "LV - Transformers",
        "LV - Substations"
    ],

    # "Other": [
    #     "FREE12"
    # ]
}
category_colors = {

    "Solar": "#81a3aaff",             
    "Wind": "#9bb17cff",                
    "Other Renewables": "#619ed0ff",    
    "Hydropower": "#a5d8f7ff",
    "Hydrogen": "#416687ff",          
    "Nuclear": "#991d51ff",             
    "Fossil Fuels ": "#4a3f69ff",   
    "Fossil Electricity": "#885ba6ff",   
    "Fossil + CCS": "#b6a1dbff",          
    "Biomass": "#2f7840ff" ,              
    "Electricity Storage": "#da607eff",   
    "Electricity Grid": "#eccbe7ff",      
    # "Other": "#f8c6b2"                  
}


stack_order = [
    "Fossil Fuels ",
    "Fossil Electricity",
    "Fossil + CCS",
    "Electricity Storage",
    "Electricity Grid",
    "Nuclear",
    "Other Renewables",
    "Hydropower",
    "Biomass",
    "Wind",
    "Hydrogen",
    "Solar",
    # "Other"  
]

import matplotlib.pyplot as plt

unit = "megaton"

grouped = {
    category: (
        all_sectors_all_inflow
        .sel(Type=techs)
        # .sel(material=["aluminium", "copper", "steel", "concrete"])
        .sel(material=["concrete"])
        # .sel(material=["steel"])
        # .sel(material=["aluminium", "copper"])
        .sum(dim=["Region", "material","Type"])
        .sel(time=slice(1990, 2100))
        .pint.to(unit)
    )
    for category, techs in categories.items()
}

df_all = pd.DataFrame({
    name: da.to_pandas()
    for name, da in grouped.items()
})

df_all = df_all[stack_order]
ax = df_all.plot.area(stacked=True, figsize=(8, 6), color=[category_colors[c] for c in df_all.columns])

ax.set_title(
    "Contribution of each type of ALL Energy Infrastructure to Inflow - SSP1 CP Other"
)

ax.set_xlabel("Time", fontsize=19,labelpad=20, fontweight='medium')
ax.set_ylabel(f"Stock ({unit})", fontsize=19,labelpad=20, fontweight='medium')

ax.grid(True, alpha=0.07)
handles, labels = ax.get_legend_handles_labels()

ax.legend(
    handles[::-1],
    labels[::-1],
    title="Category"
)

# ax.spines['top'].set_visible(False)
# ax.spines['right'].set_visible(False)
ax.margins(x=0)
ax.legend().remove()
# ax.set_ylim(0, 63_000)

ax.tick_params(axis='both', labelsize=14)

plt.show()

In [ ]:
logging.getLogger('matplotlib').setLevel(logging.ERROR)
from imagematerials.concepts import create_fossil_fuel_graph

kgraph_fossil_fuel = create_fossil_fuel_graph()

stocks_ff_extr = model.ff_extr["stock_by_cohort_materials"]
stocks_ff_proc = model.ff_proc["stock_by_cohort_materials"]
stocks_ff_trp = model.ff_trp["stock_by_cohort_materials"]
stocks_ff_pipe = model.ff_pipe["stock_by_cohort_materials"]
stocks_ff_str = model.ff_str["stock_by_cohort_materials"]
all_sectors_ff_stocks = xr.concat([stocks_ff_extr, stocks_ff_proc, stocks_ff_trp, stocks_ff_pipe, stocks_ff_str], dim="Type")

test_all_FF_stocks = kgraph_fossil_fuel.aggregate_sum(all_sectors_ff_stocks, ["Coal", "Oil", "Gas"])
test_all_FF_stocks

# Stackedplot for oil stocks by material 
import matplotlib.pyplot as plt

colors = {
    "steel": "#d2edf3ff",
    "concrete": "#353992ff",
    "aluminium": "#f8c3daff",
    "copper": "#f05135ff"
}

unit = "megaton"

oil_materials = (
    test_all_FF_stocks
    # .sel(Type="Oil")
    .sum(dim=["Region", "Type"])
    .sel(material=["aluminium", "copper", "steel", "concrete"])
    .sel(Time=slice(1990, 2100))
    .pint.to(unit)
)

df_oil = oil_materials.to_pandas()

ax = df_oil.plot.area(stacked=True, figsize=(8, 6), color=[colors[col] for col in df_oil.columns])

ax.set_title("Material Stock embedded in Fossil Fuel Infrastructure - SSP3 No Policy")
ax.set_xlabel("Time")
ax.set_ylabel(f"Stock ({unit})")

ax.grid(True, alpha=0.07)

ax.legend(title="Material", loc="upper left")

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.margins(x=0)

plt.show()
 


In [ ]:
logging.getLogger('matplotlib').setLevel(logging.ERROR)

stocks_ff_extr = model.ff_extr["stock_by_cohort_materials"]
stocks_ff_proc = model.ff_proc["stock_by_cohort_materials"]
stocks_ff_trp = model.ff_trp["stock_by_cohort_materials"]
stocks_ff_pipe = model.ff_pipe["stock_by_cohort_materials"]
stocks_ff_str = model.ff_str["stock_by_cohort_materials"]
all_sectors_ff_stocks = xr.concat([stocks_ff_extr, stocks_ff_proc, stocks_ff_trp, stocks_ff_pipe, stocks_ff_str], dim="Type")

test_all_FF_stocks = kgraph_fossil_fuel.aggregate_sum(all_sectors_ff_stocks, ["Coal", "Oil", "Gas"])
test_all_FF_stocks

# Stackedplot for gas stocks by material 
import matplotlib.pyplot as plt


colors = {
    "steel": "#d2edf3ff",
    "concrete": "#353992ff",
    "aluminium": "#f8c3daff",
    "copper": "#f05135ff",
}


unit = "megaton"

gas_materials = (
    test_all_FF_stocks
    .sel(Type="Gas")
    .sum(dim=["Region"])
    .sel(material=["aluminium", "copper", "steel", "concrete"])
    .sel(Time=slice(1990, 2100))
    .pint.to(unit)
)

df_gas = gas_materials.to_pandas()

ax = df_gas.plot.area(stacked=True, figsize=(8, 6), color=[colors[col] for col in df_gas.columns])


ax.set_title("Material Stock embedded in Gas Infrastructure - SSP3 No Policy")
ax.set_xlabel("Time")
ax.set_ylabel(f"Stock ({unit})")
ax.grid(True, alpha=0.1)
ax.legend(title="Material", loc="upper left")

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.margins(x=0)

plt.show()
 


In [ ]:
#Line plot for material stocks for all fossil fuels (coal, oil, gas) - stacked plot by material for each fuel type and also total stock by material for all fossil fuels.
materials_plot = test_all_FF_stocks.sum(dim=["Type", "Region"]).pint.to("megaton").sel(Time=slice(1990, 2100))

materials_plot.plot.line(
    x="Time",
    hue="material",
    figsize=(8, 6)
)
unit = "megaton"
plt.ylabel(f"Stock ({unit})", fontsize=14)
plt.title("Fossil Fuel Stocks by Material - SSP2 Baseline")
plt.grid(True)
plt.show()


In [ ]:
#Line plot for material stocks for all fossil fuels (coal, oil, gas) - stacked plot by material for each fuel type and also total stock by material for all fossil fuels.
import matplotlib.pyplot as plt

all_sectors_ff_stocks = xr.concat([stocks_ff_extr, stocks_ff_proc, stocks_ff_trp, stocks_ff_pipe, stocks_ff_str], dim="Type")
test_all_FF_stocks = kgraph_fossil_fuel.aggregate_sum(all_sectors_ff_stocks, ["Coal", "Oil", "Gas"])

materials_barchart = (
    test_all_FF_stocks
    .sum(dim=["Type","Region"])
    .pint.to("megaton")
    .sel(Time=[2020,2050,2100])
    .sel(material=["steel", "concrete"])
)

colors = {
    "steel": "#d2edf3ff",
    "concrete": "#353992ff"
}

df = materials_barchart.to_pandas()

ax = df.plot.bar(
    stacked=True,
    figsize=(6, 8),
    color=[colors[col] for col in df.columns]
)

unit = "megaton"
plt.ylabel(f"Stock ({unit})", fontsize=14)
plt.title("Material Stocks embedded in Fossil Fuel Infrastructure - 2020 - SSP3 NP")
plt.grid(True, axis="y", alpha=0.3)
ax.set_ylim(0, 3000)

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
#Line plot for material stocks for all fossil fuels (coal, oil, gas) - stacked plot by material for each fuel type and also total stock by material for all fossil fuels.
import matplotlib.pyplot as plt

all_sectors_ff_stocks = xr.concat([stocks_ff_extr, stocks_ff_proc, stocks_ff_trp, stocks_ff_pipe, stocks_ff_str], dim="Type")
test_all_FF_stocks = kgraph_fossil_fuel.aggregate_sum(all_sectors_ff_stocks, ["Coal", "Oil", "Gas"])

materials_barchart = (
    test_all_FF_stocks
    .sum(dim=["Type","Region"])
    .pint.to("megaton")
    .sel(Time=[2020,2050,2100])
    .sel(material=["aluminium", "copper", "pe", "pvc", "silicon", "zinc"])
)

colors = {
    "aluminium": "#f8c3daff",
    "copper": "#f05135ff",
    "pe": "#5F5EA3",
    "pvc": "#8B4513",
    "silicon": "#F17474",
    "zinc": "#2F0E68"
}
df = materials_barchart.to_pandas()

ax = df.plot.bar(
    stacked=True,
    figsize=(6, 8),
    color=[colors[col] for col in df.columns]
)

unit = "megaton"
plt.ylabel(f"Stock ({unit})", fontsize=14)
plt.title("Material Stocks embedded in Fossil Fuel Infrastructure - 2020 - SSP1 CP")
plt.grid(True, axis="y", alpha=0.3)
ax.set_ylim(0, 12)

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
#Line plot for material stocks for all fossil fuels (coal, oil, gas) - stacked plot by material for each fuel type and also total stock by material for all fossil fuels.
import matplotlib.pyplot as plt

all_sectors_ff_stocks = xr.concat([stocks_ff_extr, stocks_ff_proc, stocks_ff_trp, stocks_ff_pipe, stocks_ff_str], dim="Type")
test_all_FF_stocks = kgraph_fossil_fuel.aggregate_sum(all_sectors_ff_stocks, ["Coal", "Oil", "Gas"])

stocks_2020 = (
    test_all_FF_stocks
    .sum(dim=["Region"])
    .pint.to("megaton")
    .sel(Time=1990)
)


df = stocks_2020.to_pandas().T  

material_colors = {
    "aluminium": "#74B586",
    "copper": "#D5573B",
    "pe": "#5F5EA3",
    "pvc": "#8B4513",
    "silicon": "#F17474",
    "zinc": "#2F0E68",
    "steel": "#A5D8FF",
    "concrete": "#6B4E71",
}

colors = [material_colors[m] for m in df.index]

ax = df.T.plot(
    kind="bar",
    stacked=True,
    figsize=(4, 6),
    color=colors
)

plt.ylabel("Stock (megaton)", fontsize=14)
plt.title("Material Stocks by Fuel Type (1990) - SSP1 CP", fontsize=16)
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.ylim(0, 2200)
plt.tight_layout()
plt.show()


In [ ]:
all_output["SSP2"].ff_extr.keys()
# all_output["SSP2"].ff_extr.keys()

## Plot materials

### s: variables for plots

In [ ]:
logging.getLogger('matplotlib').setLevel(logging.ERROR)

# you can also save the steps in between in a variable to avoid repeating the same things, for example:
# save the model output of a specific scenario
new_output = all_output["SSP1_climate_policy"]

coal_extraction_types = ["Coal Opencast", "Coal Underground"]
coal_processing_types = ["Coal Preparation"]
coal_transport_types = ["Coal Inland Ship", "Coal Ocean Ship", "Coal Rail", "Coal Truck"]

gas_extraction_types = ["Gas Offshore", "Gas Onshore"]
gas_processing_types = ["Gas Processing"]
gas_transport_types = ["Gas Inland Ship", "Gas Ocean Ship", "Gas Rail", "Gas Truck"]
gas_pipeline_types = ["Gas Distribution Pipeline", "Gas Transmission Pipeline"] 

oil_extraction_types = ["Oil Offshore", "Oil Onshore"]
oil_transport_types = ["Oil Inland Ship", "Oil Ocean Ship", "Oil Rail", "Oil Truck"]
oil_pipeline_types = ["Oil Onshore Crude Pipeline", "Oil Onshore Product Pipeline", "Oil Offshore Crude Pipeline", "Oil Offshore Product Pipeline"] 
oil_processing_types = ["Oil Refinery"]
oil_storage_types = ["Oil Storage"]


# save a certain variable
# gen_inflow_global = model_scenario_x.elc_gen["inflow"].to_array().sum(["Region"])

# #To change a unit do this:
# var = model_scenario_x.elc_gen["inflow_materials"].to_array().pint.to("megaton")

# # If you automatically want to have the correct unit printed in a figure you can do:
# unit = "megaton"
# var = model_scenario_x.elc_gen["inflow_materials"].to_array().pint.to(unit)

# fig, ax = plt.subplots(figsize=(8,6))

# plt.ylabel(f"material inflow ({unit})", fontsize=14)


### Line graph for one material

In [ ]:
logging.getLogger('matplotlib').setLevel(logging.ERROR)

#the key for the stock by cohort materials is already in xarray format, so we can directly select the material and region and plot it.

#plot for steel in all regions and types for each sector plotted on the same graph to compare the sectors.
extr = all_output["SSP2"].ff_extr["stock_by_cohort_materials"].sel(material="steel").sel(Time=slice(1990, 2050)).sum(dim="Type").sum(dim="Region").pint.to("megaton")
extr.plot(label="Extraction")

pipe = all_output["SSP2"].ff_pipe["stock_by_cohort_materials"].sel(material="steel").sel(Time=slice(1990, 2050)).sum(dim="Type").sum(dim="Region").pint.to("megaton")
pipe.plot(label="Pipeline")

proc = all_output["SSP2"].ff_proc["stock_by_cohort_materials"].sel(material="steel").sel(Time=slice(1990, 2050)).sum(dim="Type").sum(dim="Region").pint.to("megaton")
proc.plot(label="Processing")

trp = all_output["SSP2"].ff_trp["stock_by_cohort_materials"].sel(material="steel").sel(Time=slice(1990, 2050)).sum(dim="Type").sum(dim="Region").pint.to("megaton")
trp.plot(label="Transportation")

str = all_output["SSP2"].ff_str["stock_by_cohort_materials"].sel(material="steel").sel(Time=slice(1990, 2050)).sum(dim="Type").sum(dim="Region").pint.to("megaton")
str.plot(label="Storage")

# Sum of all
total = extr + pipe + proc + trp + str
total.plot(label="Total", linewidth=2, linestyle="--")
plt.title("Steel Stock by Cohort Materials - Fossil Fuels - SSP2 FUMA")
plt.xlabel("Time")
unit = "megaton"
plt.ylabel(f"Stock ({unit})", fontsize=14)
plt.legend()
plt.grid(True)
plt.ylim(0, 1800) 

plt.legend()

print(total.sel(Time=[2019, 2050]).to_series())

### Fill in the material and unit stacked line graph for stock

In [ ]:
logging.getLogger('matplotlib').setLevel(logging.ERROR)

material = "concrete"
unit = "megaton"
firstplotyear = 1990
lastplotyear = 2100
scenario_output = new_output = all_output["SSP1_climate_policy"]

#plot for steel in all regions and types for each sector plotted on the same graph to compare the sectors.
gas_extr = scenario_output.ff_extr["stock_by_cohort_materials"].sel(material=material, Type=gas_extraction_types, Time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
gas_pipe = scenario_output.ff_pipe["stock_by_cohort_materials"] .sel(material=material, Type=gas_pipeline_types, Time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
gas_proc = scenario_output.ff_proc["stock_by_cohort_materials"].sel(material=material, Type=gas_processing_types, Time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
gas_trp = scenario_output.ff_trp["stock_by_cohort_materials"].sel(material=material, Type=gas_transport_types, Time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
oil_extr = scenario_output.ff_extr["stock_by_cohort_materials"] .sel(material=material, Type=oil_extraction_types, Time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
oil_pipe = scenario_output.ff_pipe["stock_by_cohort_materials"].sel(material=material, Type=oil_pipeline_types, Time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
oil_proc = scenario_output.ff_proc["stock_by_cohort_materials"] .sel(material=material, Type=oil_processing_types, Time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
oil_trp = scenario_output.ff_trp["stock_by_cohort_materials"] .sel(material=material, Type=oil_transport_types, Time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
oil_str = scenario_output.ff_str["stock_by_cohort_materials"] .sel(material=material, Type=oil_storage_types, Time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
coal_extr = scenario_output.ff_extr["stock_by_cohort_materials"].sel(material=material, Type=coal_extraction_types, Time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
coal_proc = scenario_output.ff_proc["stock_by_cohort_materials"].sel(material=material, Type=coal_processing_types, Time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
coal_trp = scenario_output.ff_trp["stock_by_cohort_materials"] .sel(material=material, Type=coal_transport_types, Time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)

gas_total = gas_extr + gas_pipe + gas_trp + gas_proc
coal_total = coal_extr + coal_proc + coal_trp
oil_total = oil_extr + oil_pipe + oil_proc + oil_trp + oil_str
total_overall = gas_total + coal_total + oil_total

fig, ax = plt.subplots(figsize=(8,6))

ax.stackplot(
    gas_total.Time.values,
    coal_total.values,   # bottom
    oil_total.values,    # middle
    gas_total.values,    # top
    labels=["Coal", "Oil", "Gas"],
    colors=["#4A5759", "#edafb8ff", "#5089A5"]
)

ax.set_xlabel("Time", fontsize=14)
ax.set_ylabel(f"Stock ({unit})", fontsize=14)
ax.set_title(f"{material.title()} Stock embedded in Fossil Fuel Infrastructure - SSP1 Climate Policy", fontweight="bold")

ax.legend(loc="upper left")
ax.grid(alpha=0.3)

# print(total_overall.sel(Time=[2019, 2050]).to_series())


### Fill in the material for inflow/outflow

In [ ]:
# Test idea for final figures 

material = "steel"
unit = "megaton"
firstplotyear = 1990
lastplotyear = 2100
scenario_output = new_output = all_output["SSP3_no_policy"]


#plot for steel in all regions and types for each sector plotted on the same graph to compare the sectors.
gas_extr = scenario_output.ff_extr["inflow_materials"].to_array().sel(material=material, Type=gas_extraction_types, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
gas_pipe = scenario_output.ff_pipe["inflow_materials"].to_array().sel(material=material, Type=gas_pipeline_types, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
oil_storage = scenario_output.ff_str["inflow_materials"].to_array().sel(material=material,Type=oil_storage_types,time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
gas_proc = scenario_output.ff_proc["inflow_materials"].to_array().sel(material=material, Type=gas_processing_types, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
gas_trp = scenario_output.ff_trp["inflow_materials"].to_array().sel(material=material, Type=gas_transport_types, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
oil_extr = scenario_output.ff_extr["inflow_materials"] .to_array().sel(material=material, Type=oil_extraction_types, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
oil_pipe = scenario_output.ff_pipe["inflow_materials"].to_array().sel(material=material, Type=oil_pipeline_types, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
oil_proc = scenario_output.ff_proc["inflow_materials"].to_array().sel(material=material, Type=oil_processing_types, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
oil_trp = scenario_output.ff_trp["inflow_materials"] .to_array().sel(material=material, Type=oil_transport_types, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
coal_extr = scenario_output.ff_extr["inflow_materials"].to_array().sel(material=material, Type=coal_extraction_types, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
coal_proc = scenario_output.ff_proc["inflow_materials"].to_array().sel(material=material, Type=coal_processing_types, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
coal_trp = scenario_output.ff_trp["inflow_materials"] .to_array().sel(material=material, Type=coal_transport_types, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)


gas_total = gas_extr + gas_pipe + gas_trp + gas_proc
coal_total = coal_extr + coal_proc + coal_trp
oil_total = oil_extr + oil_pipe + oil_proc + oil_trp + oil_storage
total_overall = gas_total + coal_total + oil_total

#plot for steel in all regions and types for each sector plotted on the same graph to compare the sectors.
gas_extr_out = scenario_output.ff_extr["outflow_by_cohort_materials"].to_array().sel(material=material, Type=gas_extraction_types, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
gas_pipe_out = scenario_output.ff_pipe["outflow_by_cohort_materials"].to_array().sel(material=material, Type=gas_pipeline_types, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
oil_storage_out = scenario_output.ff_str["outflow_by_cohort_materials"].to_array().sel(material=material,Type=oil_storage_types,time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
gas_proc_out = scenario_output.ff_proc["outflow_by_cohort_materials"].to_array().sel(material=material, Type=gas_processing_types, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
gas_trp_out = scenario_output.ff_trp["outflow_by_cohort_materials"].to_array().sel(material=material, Type=gas_transport_types, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
oil_extr_out = scenario_output.ff_extr["outflow_by_cohort_materials"] .to_array().sel(material=material, Type=oil_extraction_types, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
oil_pipe_out = scenario_output.ff_pipe["outflow_by_cohort_materials"].to_array().sel(material=material, Type=oil_pipeline_types, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
oil_proc_out = scenario_output.ff_proc["outflow_by_cohort_materials"].to_array().sel(material=material, Type=oil_processing_types, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
oil_trp_out = scenario_output.ff_trp["outflow_by_cohort_materials"] .to_array().sel(material=material, Type=oil_transport_types, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
coal_extr_out = scenario_output.ff_extr["outflow_by_cohort_materials"].to_array().sel(material=material, Type=coal_extraction_types, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
coal_proc_out = scenario_output.ff_proc["outflow_by_cohort_materials"].to_array().sel(material=material, Type=coal_processing_types, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)
coal_trp_out = scenario_output.ff_trp["outflow_by_cohort_materials"] .to_array().sel(material=material, Type=coal_transport_types, time=slice(firstplotyear, lastplotyear)).sum(dim="Type").sum(dim="Region").pint.to(unit)

# Sum of all
gas_total_out = gas_extr_out + gas_pipe_out + gas_proc_out + gas_trp_out
coal_total_out = coal_extr_out + coal_proc_out + coal_trp_out
oil_total_out = oil_extr_out + oil_pipe_out + oil_proc_out+ oil_trp_out + oil_storage_out
total_overall_out = gas_total_out + coal_total_out + oil_total_out


fig, ax = plt.subplots(figsize=(10,6))

ax.stackplot(
    gas_total.time.values,
    coal_total.values,   # bottom
    gas_total.values, 
    oil_total.values,    # middle
    labels=["Coal", "Gas", "Oil"],
    colors=["#4A5759", "#5089A5", "#edafb8ff"]
)

ax.stackplot(
    gas_total_out.time.values,
    -coal_total_out.values, 
    -gas_total_out.values,  # bottom
    -oil_total_out.values,    # middle
    # labels=["Coal", "Oil", "Gas"],
    colors=["#758082", "#6CAAC9", "#fbd4daff"]
)

ax.set_xlabel("time", fontsize=14)
ax.set_ylabel(f"Inflow & Outflow ({unit})", fontsize=14)
ax.set_title(f"{material.title()} Inflow & Outflow- SSP3 NP", fontweight="bold")

ax.legend(loc="upper left")
ax.grid(alpha=0.3)

print(total_overall.sel(time=[2019, 2050]).to_series())

#line charts comparing total stock, inflow, outflow for each material
# Line chart comparing scenarios for total stock, inflow, outflow for each material/each fuel type 
# run and create totals for each materials and then make a figure with those totals compared?


